<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Régression linéaire - RMSE public : 
</div>
<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des bilbiothèques
</div>

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import root_mean_squared_error
import gc
from vacances_scolaires_france import SchoolHolidayDates
from jours_feries_france import JoursFeries

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des documents de données temporelles et de test
</div>

In [2]:
stations_toulouse = pd.read_parquet("stations_toulouse.parquet") # Données géographiques (Colonnes : 'station', 'longitude', 'latitude', 'station_id')
history_toulouse_train = pd.read_parquet("history_toulouse_train.parquet")
history_toulouse_test = pd.read_parquet("history_toulouse_test.parquet")

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Données du train
</div>
On transforme les indexe en colonnes pour plus de praticité

In [3]:
train = history_toulouse_train.reset_index()

On extrait tous les indexe station et commit_at_index sous forme de liste

In [4]:
station_index = history_toulouse_test.index.get_level_values("station")
commit_at_index = history_toulouse_test.index.get_level_values("commit_at")

Prise en compte de la date

In [5]:
train["annee"] = train["commit_at"].dt.year
train["mois"] = train["commit_at"].dt.month
train["jour"] = train["commit_at"].dt.dayofweek
train["weekend"] = (train["jour"] >= 5).astype(int)
train["heure"] = train["commit_at"].dt.hour
train["minute"] = train["commit_at"].dt.minute

On encode les variables temporelles de manière cyclique pour qu'il n'y ait pas de rupture (ex : 23h -> 0h)

In [6]:
train["mois_sin"] = np.sin(2 * np.pi * train["mois"] / 12)
train["mois_cos"] = np.cos(2 * np.pi * train["mois"] / 12)
train["jour_sin"] = np.sin(2 * np.pi * train["jour"] / 7)
train["jour_cos"] = np.cos(2 * np.pi * train["jour"] / 7)
train["heure_sin"] = np.sin(2 * np.pi * train["heure"] / 24)
train["heure_cos"] = np.cos(2 * np.pi * train["heure"] / 24)
train["minute_sin"] = np.sin(2 * np.pi * train["minute"] / 60)
train["minute_cos"] = np.cos(2 * np.pi * train["minute"] / 60)

Prise en compte de la géolocalisation

In [7]:
train = train.merge(stations_toulouse, on="station", how="left")

Prise en compte du nombre total de stations

In [8]:
train["total"] = train["bikes"] + train["stands"]

Prise en compte du taux d'occupation de la station + on évite la division par 0

In [9]:
train["occupation"] = train["bikes"] / train["total"].replace(0, 1)

Prise en compte des vacances scolaires (Toulouse est en zone C)

In [10]:
# On extrait toutes les dates de train et test
dates_train = train["commit_at"].dt.date.unique()
dates_test = commit_at_index.date
# On en extrait toutes les dates différentes
toutes_dates = pd.Series(list(dates_train) + list(dates_test)).unique()
# Pour chaque date on détecte si Toulouse est en vacance ou non
d = SchoolHolidayDates()
vacances_par_date = {date: d.is_holiday_for_zone(date, 'C') for date in toutes_dates}
train["vacances"] = train["commit_at"].dt.date.map(vacances_par_date).astype(int)

Prise en compte des jours fériés

In [11]:
# Toutes les années dans train union (|) test
annees = set(train["annee"].unique()) | set(commit_at_index.year.unique())
# On construit un set de tous les jours feriés de toutes les années détectées
tous_feries = set()
for annee in annees:
    tous_feries.update(JoursFeries.for_year(int(annee)).values())
# On écrit le résultat dans train
train["ferie"] = train["commit_at"].dt.date.isin(tous_feries).astype(int)

Données utiles pour l'entrainement du modèle

In [12]:
colonnes_train = ["bikes",
                  "annee",
                  "mois_sin", "mois_cos",
                  "jour_sin", "jour_cos",
                  "weekend",
                  "heure_sin", "heure_cos",
                  "minute_sin", "minute_cos",
                  "longitude", "latitude",
                  "total", "occupation",
                 "vacances", "ferie"]

On crée un dataframe reliant (station, date) au nombre de vélos

In [13]:
date_velo = train.set_index(["station", "commit_at"])["bikes"]

On associe chaque nom de colonne à une valeur numérique

In [14]:
decalage = {
    "bikes_+15min": 15,
    "bikes_+30min": 30,
    "bikes_+45min": 45,
    "bikes_+60min": 60,
    "bikes_+75min": 75,
    "bikes_+90min": 90,
    "bikes_+105min": 105,
    "bikes_+120min": 120,
}

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Données du test
</div>
On récupère le nombre de vélos à l'instant zero du test

In [15]:
instant_zero = list(zip(station_index, commit_at_index))
bikes_train = history_toulouse_train["bikes"]
stands_train = history_toulouse_train["stands"]

On localise chaque index (station, date) pour trouver le nombre de vélos et d'emplacement vides à l'instant zero du test

In [16]:
bikes_zero = pd.Series([bikes_train.loc[instant] for instant in instant_zero], index=history_toulouse_test.index)
stands_zero = pd.Series([stands_train.loc[instant] for instant in instant_zero], index=history_toulouse_test.index)

On récupère la géolocalisation de chaque station du test <br>
On définit la colonne "station" comme étant l'indexe

In [17]:
stations_pos = stations_toulouse.set_index("station")

On localise chaque station pour trouver leur géolocalisation

In [18]:
longitude_test = [stations_pos.loc[station, "longitude"] for station in station_index]
latitude_test = [stations_pos.loc[station, "latitude"] for station in station_index]

On crée un dataframe avec toutes les données du test dont on aura besoin pour prédire. <br>
Il faut qu'il y ait les mêmes données que celles utilisées pour l'entrainement

In [19]:
donnees_test = pd.DataFrame({
    "bikes": bikes_zero,
    "annee": commit_at_index.year,
    "mois_sin": np.sin(2 * np.pi * commit_at_index.month / 12),
    "mois_cos": np.cos(2 * np.pi * commit_at_index.month / 12),
    "jour_sin": np.sin(2 * np.pi * commit_at_index.dayofweek / 7),
    "jour_cos": np.cos(2 * np.pi * commit_at_index.dayofweek / 7),
    "weekend": (commit_at_index.dayofweek >= 5).astype(int),
    "heure_sin": np.sin(2 * np.pi * commit_at_index.hour / 24),
    "heure_cos": np.cos(2 * np.pi * commit_at_index.hour / 24),
    "minute_sin": np.sin(2 * np.pi * commit_at_index.minute / 60),
    "minute_cos": np.cos(2 * np.pi * commit_at_index.minute / 60),
    "longitude": longitude_test,
    "latitude": latitude_test,
    "total": bikes_zero + stands_zero,
    "occupation": bikes_zero / (bikes_zero + stands_zero).replace(0, 1),
    "vacances": pd.Series(commit_at_index.date, index=history_toulouse_test.index).map(vacances_par_date).astype(int),
    "ferie": pd.Series(commit_at_index.date, index=history_toulouse_test.index).isin(tous_feries).astype(int),
}, index=history_toulouse_test.index)

On liste les colonnes réellement utiles à l'entrainement

In [20]:
colonnes_necessaires = list(dict.fromkeys(["station", "commit_at"] + colonnes_train))

On en déduit celles qui ne le sont pas

In [21]:
colonnes_a_supprimer = [c for c in train.columns if c not in colonnes_necessaires]

On supprime les colonnes non-nécessaires (pour alléger le copy dans la boucle d'entrainement

In [22]:
train = train.drop(columns=colonnes_a_supprimer)

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Entrainement et prédictions
</div>
On entraine un modèle par décalage horaire et on le fait prédire

In [ ]:
for col, h in decalage.items():
    print(h/1.2, "%")
    # On crée une copie du dataframe pour chaque décalage horaire
    avec_futur = train.copy()
    # On crée une colonne avec l'horaire à prédire
    avec_futur["commit_at_cible"] = avec_futur["commit_at"] + pd.Timedelta(minutes=h)
    # On trouve le nombre de vélos en prenant en compte le décalage horaire
    indexes_cibles = list(zip(avec_futur["station"], avec_futur["commit_at_cible"]))
    avec_futur["bikes_futur"] = [date_velo.get(indexe) for indexe in indexes_cibles]
    # On retire les lignes dans lesquelles le nombre de vélos n'est pas connu
    avec_futur = avec_futur.dropna(subset=["bikes_futur"])

    # Prédictions
    # On renseigne les données d'entrainement et leur réponse
    X = avec_futur[colonnes_train]
    y = avec_futur["bikes_futur"]
    
    # On prépare les données en créant des x² et plus
    polynomial_features = PolynomialFeatures(degree = 2)
    X_TRANSF = polynomial_features.fit_transform(X)

    model = LinearRegression()
    # On entraine le modèle
    model.fit(X_TRANSF, y)
    # On prédit
    pred = model.predict(donnees_test[colonnes_train])
    # On empêche les prédictions d'être négatives ou de prédire plus de vélos que le nombre de places totales
    pred = np.clip(pred, 0, donnees_test["total"].values).round().astype(int)
    # On écrit les prédictions dans history_toulouse_test
    history_toulouse_test[col] = pred

    # On fait une prédiction sur les données d'entrainement pour voir avoir une estimation de la performance du modèle ELLE SERA TROP OPTIMISTE DANS TOUS LES CAS
    pred_train = model.predict(X)
    pred_train = np.clip(pred_train, 0, X["total"].values).round().astype(int)
    # On affiche une mesure de performance
    print(f"{col} — RMSE sur train : {root_mean_squared_error(y, pred_train):.4f}\n")

    # On supprime les variables qui prennent de la place en mémoire
    del avec_futur, X, y, model, pred_train, pred
    # On force le ramasse-miettes de python à passer pour effectivement libérer la mémoire
    gc.collect()

12.5 %


<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Sauvegarde du résultat
</div>

In [ ]:
history_toulouse_test.to_parquet("BA_prediction_200_RP.parquet", index=True)
history_toulouse_test